# unittest in Python: From Basics to Intermediate (with Trading-Flavored Examples)

This notebook is a **from-scratch to intermediate** guide to Python’s built-in
**`unittest`** framework.

We’ll use **trading-flavored** examples (price functions, order logic, risk checks)
to show how to structure tests, run them, and extend the framework.

### Contents

- [1. Why unittest and When to Use It?](#1-why-unittest-and-when-to-use-it)
- [2. Basic Structure: TestCase, Assertions, and Test Discovery](#2-basic-structure-testcase-assertions-and-test-discovery)
- [3. Testing Pure Functions (Pricing, PnL, Indicators)](#3-testing-pure-functions-pricing-pnl-indicators)
- [4. setUp, tearDown, and Test Fixtures](#4-setup-teardown-and-test-fixtures)
- [5. Testing Classes and State (Orders, Positions)](#5-testing-classes-and-state-orders-positions)
- [6. Mocking and Patching (HTTP Calls, Time, Randomness)](#6-mocking-and-patching-http-calls-time-randomness)
- [7. Parameterized Tests and SubTests](#7-parameterized-tests-and-subtests)
- [8. Organizing Tests in a Project](#8-organizing-tests-in-a-project)
- [9. Running Tests: CLI, Verbosity, and Integration with pytest](#9-running-tests-cli-verbosity-and-integration-with-pytest)
- [10. Best Practices for Quant/Trading Code](#10-best-practices-for-quanttrading-code)

## 1. Why `unittest` and When to Use It?

Python’s **`unittest`** module is the **standard library** testing framework:

- No extra dependencies; always available.
- xUnit-style: `TestCase` classes, methods like `test_something`, assertion methods.
- Works well for unit and small integration tests.

Even if you eventually use **pytest**, understanding `unittest` helps because:

- pytest can run `unittest` test cases.
- Many existing codebases and libraries still use it.

We’ll focus on patterns useful for **quant/trading code**: verifying pricing functions,
order/position logic, risk checks, and external API interactions.

In [4]:
# 1.1 Import unittest

import unittest

print("unittest module:", unittest)

unittest module: <module 'unittest' from 'c:\\DEV\\Python\\Lib\\unittest\\__init__.py'>


## 2. Basic Structure: TestCase, Assertions, and Test Discovery

Key pieces:

- **`unittest.TestCase`**: base class for test classes.
- Test methods must start with **`test_`** to be discovered.
- Use assertion methods like `assertEqual`, `assertTrue`, `assertAlmostEqual`,
  `assertRaises`, etc.
- Tests live in files named `test_*.py` or `*_test.py` typically.

Minimal example in `test_example.py`:

```python
import unittest

class TestMath(unittest.TestCase):
    def test_add(self):
        self.assertEqual(1 + 1, 2)

if __name__ == "__main__":
    unittest.main()
```

Run with:

```bash
python -m unittest test_example.py
```

In [ ]:
# 2.1 Minimal unittest example executed inline

class TestMath(unittest.TestCase):
    def test_add(self):
        self.assertEqual(1 + 1, 2)


# Run the tests in this cell only
suite = unittest.TestLoader().loadTestsFromTestCase(TestMath)
result = unittest.TextTestRunner(verbosity = 2).run(suite)
print("Tests run:", result.testsRun, "Failures:", len(result.failures))

test_add (__main__.TestMath.test_add) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


Tests run: 1 Failures: 0


## 3. Testing Pure Functions (Pricing, PnL, Indicators)

Pure functions are easiest to test: given inputs → expected output.

We’ll define a tiny `pricing.py` module (in this notebook as simple functions):

- `mid_price(bid, ask)`
- `pnl(entry, exit, qty)`

Then we’ll write `TestPricing` to cover edge cases: invalid inputs, short vs long,
zero quantity, etc.

In [6]:
# 3.1 Pricing helper functions (would live in pricing.py)

from typing import Optional


def mid_price(bid: float, ask: float) -> Optional[float]:
    """Return mid price or None if inputs invalid.

    In real code you might raise instead; here we use None to keep it simple.
    """
    if bid <= 0 or ask <= 0 or bid >= ask:
        return None
    return (bid + ask) / 2.0


def pnl(entry: float, exit: float, qty: float) -> float:
    """Simple PnL: (exit - entry) * qty.

    Positive qty = long; negative qty = short.
    """
    return (exit - entry) * qty


class TestPricing(unittest.TestCase):
    def test_mid_price_normal(self):
        self.assertEqual(mid_price(99.0, 101.0), 100.0)

    def test_mid_price_invalid(self):
        self.assertIsNone(mid_price(101.0, 99.0))  # crossed market
        self.assertIsNone(mid_price(-1.0, 101.0))

    def test_pnl_long_vs_short(self):
        self.assertAlmostEqual(pnl(100.0, 110.0, 10), 100.0)  # long
        self.assertAlmostEqual(pnl(100.0, 90.0, -10), 100.0)   # short


suite = unittest.TestLoader().loadTestsFromTestCase(TestPricing)
unittest.TextTestRunner(verbosity=2).run(suite)

test_mid_price_invalid (__main__.TestPricing.test_mid_price_invalid) ... ok
test_mid_price_normal (__main__.TestPricing.test_mid_price_normal) ... ok
test_pnl_long_vs_short (__main__.TestPricing.test_pnl_long_vs_short) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.003s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

## 4. setUp, tearDown, and Test Fixtures

`TestCase` provides hooks:

- `setUp(self)` – run **before each test method**
- `tearDown(self)` – run **after each test method**

You can also use `setUpClass` / `tearDownClass` for class-level fixtures.

We’ll simulate a tiny `Position` class that tracks quantity and average price and use
`setUp` to create a fresh instance for each test.

In [7]:
# 4.1 Position class and setUp/tearDown usage

class Position:
    def __init__(self, symbol: str):
        self.symbol = symbol
        self.quantity = 0.0
        self.avg_price = 0.0

    def apply_fill(self, side: str, qty: float, price: float) -> None:
        if qty <= 0:
            raise ValueError("qty must be positive")
        signed = qty if side.upper() == "BUY" else -qty
        new_qty = self.quantity + signed
        if new_qty == 0:
            self.quantity = 0.0
            self.avg_price = 0.0
            return
        total_value = self.avg_price * self.quantity + price * signed
        self.quantity = new_qty
        self.avg_price = total_value / self.quantity


class TestPosition(unittest.TestCase):
    def setUp(self):
        # Fresh position for each test
        self.pos = Position("AAPL")

    def tearDown(self):
        # Could close DB connections, files, etc.; here we just log
        # (in real tests you usually don't print)
        pass

    def test_apply_buy_then_sell(self):
        self.pos.apply_fill("BUY", 100, 180.0)
        self.pos.apply_fill("SELL", 50, 185.0)
        self.assertEqual(self.pos.symbol, "AAPL")
        self.assertAlmostEqual(self.pos.quantity, 50.0)
        self.assertGreater(self.pos.avg_price, 0.0)

    def test_invalid_qty_raises(self):
        with self.assertRaises(ValueError):
            self.pos.apply_fill("BUY", 0, 180.0)


suite = unittest.TestLoader().loadTestsFromTestCase(TestPosition)
unittest.TextTestRunner(verbosity=2).run(suite)

test_apply_buy_then_sell (__main__.TestPosition.test_apply_buy_then_sell) ... ok
test_invalid_qty_raises (__main__.TestPosition.test_invalid_qty_raises) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

## 5. Testing Classes and State (Orders, Positions)

For stateful classes, typical tests look like:

- Arrange: construct object(s) and set initial state.
- Act: call one or more methods.
- Assert: verify state and outputs.

We’ll add tests that check:

- Order validation logic (e.g. limit orders require a price).
- Position PnL and quantities after multiple fills.

In a real project, these classes would live in their own modules and tests in
`tests/test_orders.py`, `tests/test_positions.py`, etc.

In [8]:
# 5.1 Simple Order class + tests

class Order:
    def __init__(self, symbol: str, side: str, type_: str, quantity: float, limit_price: float | None = None):
        self.symbol = symbol
        self.side = side.upper()
        self.type = type_.upper()
        self.quantity = quantity
        self.limit_price = limit_price
        self.status = "NEW"

    def validate(self) -> None:
        if self.quantity <= 0:
            raise ValueError("quantity must be positive")
        if self.type == "LIMIT" and self.limit_price is None:
            raise ValueError("limit_price required for LIMIT orders")


class TestOrder(unittest.TestCase):
    def test_valid_limit_order(self):
        o = Order("AAPL", "buy", "limit", 100, 180.0)
        o.validate()  # should not raise

    def test_limit_order_without_price(self):
        o = Order("AAPL", "buy", "limit", 100)
        with self.assertRaises(ValueError):
            o.validate()

    def test_negative_qty(self):
        o = Order("AAPL", "buy", "market", -1)
        with self.assertRaises(ValueError):
            o.validate()


suite = unittest.TestLoader().loadTestsFromTestCase(TestOrder)
unittest.TextTestRunner(verbosity=2).run(suite)

test_limit_order_without_price (__main__.TestOrder.test_limit_order_without_price) ... ok
test_negative_qty (__main__.TestOrder.test_negative_qty) ... ok
test_valid_limit_order (__main__.TestOrder.test_valid_limit_order) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

## 6. Mocking and Patching (HTTP Calls, Time, Randomness)

For tests involving **external systems** (HTTP APIs, databases, time, randomness), you
usually **mock** them so tests are fast, deterministic, and offline.

`unittest.mock` provides:

- `patch` to temporarily replace objects (functions, methods, attributes).
- `MagicMock` / `Mock` to create fake objects with configurable behavior.

We’ll mock:

- An HTTP call that fetches a price.
- `time.time()` to make timestamps deterministic.

In [9]:
# 6.1 Mocking an HTTP client and time.time

from unittest.mock import patch, MagicMock


def fetch_price_from_api(symbol: str) -> float:
    """Example function that calls an external HTTP API.

    In real life, you'd use requests/httpx/aiohttp. Here we simulate via a helper.
    """
    import requests  # type: ignore

    url = "https://api.example.com/price"
    r = requests.get(url, params={"symbol": symbol}, timeout=5.0)
    r.raise_for_status()
    data = r.json()
    return float(data["price"])


class TestExternalDeps(unittest.TestCase):
    @patch("time.time")
    @patch("requests.get")
    def test_fetch_price_and_timestamp(self, mock_get: MagicMock, mock_time: MagicMock):
        # Configure mocks
        mock_time.return_value = 1_700_000_000.0
        mock_resp = MagicMock()
        mock_resp.json.return_value = {"price": "123.45"}
        mock_resp.raise_for_status.return_value = None
        mock_get.return_value = mock_resp

        price = fetch_price_from_api("BTCUSDT")
        self.assertAlmostEqual(price, 123.45)
        self.assertEqual(time.time(), 1_700_000_000.0)
        mock_get.assert_called_once()


suite = unittest.TestLoader().loadTestsFromTestCase(TestExternalDeps)
unittest.TextTestRunner(verbosity=2).run(suite)

test_fetch_price_and_timestamp (__main__.TestExternalDeps.test_fetch_price_and_timestamp) ... ERROR

ERROR: test_fetch_price_and_timestamp (__main__.TestExternalDeps.test_fetch_price_and_timestamp)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "c:\DEV\Python\Lib\unittest\mock.py", line 1426, in patched
    return func(*newargs, **newkeywargs)
  File "C:\Users\GSL\AppData\Local\Temp\ipykernel_54188\3279333095.py", line 33, in test_fetch_price_and_timestamp
    self.assertEqual(time.time(), 1_700_000_000.0)
                     ^^^^
NameError: name 'time' is not defined. Did you forget to import 'time'?

----------------------------------------------------------------------
Ran 1 test in 0.219s

FAILED (errors=1)


<unittest.runner.TextTestResult run=1 errors=1 failures=0>

## 7. Parameterized Tests and SubTests

`unittest` doesn’t have built-in parameterization like pytest’s `@pytest.mark.parametrize`,
but you can:

- Loop inside a test and use **`self.subTest(...)`** to separate cases.
- Or generate test methods dynamically.

Subtests make it easy to test multiple symbol/price scenarios with clear reporting
on which case failed.

In [ ]:
# 7.1 Using subTest for multiple scenarios

class TestPnLScenarios(unittest.TestCase):
    def test_pnl_scenarios(self):
        cases = [
            {"entry": 100.0, "exit": 110.0, "qty": 10, "expected": 100.0},
            {"entry": 100.0, "exit": 90.0, "qty": 10, "expected": -100.0},
            {"entry": 100.0, "exit": 110.0, "qty": -10, "expected": -100.0},
        ]
        for case in cases:
            with self.subTest(case=case):
                self.assertAlmostEqual(
                    pnl(case["entry"], case["exit"], case["qty"]),
                    case["expected"],
                )


suite = unittest.TestLoader().loadTestsFromTestCase(TestPnLScenarios)
unittest.TextTestRunner(verbosity=2).run(suite)

test_pnl_scenarios (__main__.TestPnLScenarios.test_pnl_scenarios) ... 
  test_pnl_scenarios (__main__.TestPnLScenarios.test_pnl_scenarios) (case={'entry': 100.0, 'exit': 110.0, 'qty': -10, 'expected': -1000.0}) ... FAIL

FAIL: test_pnl_scenarios (__main__.TestPnLScenarios.test_pnl_scenarios) (case={'entry': 100.0, 'exit': 110.0, 'qty': -10, 'expected': -1000.0})
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\GSL\AppData\Local\Temp\ipykernel_54188\649832631.py", line 12, in test_pnl_scenarios
    self.assertAlmostEqual(
    ~~~~~~~~~~~~~~~~~~~~~~^
        pnl(case["entry"], case["exit"], case["qty"]),
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        case["expected"],
        ^^^^^^^^^^^^^^^^^
    )
    ^
AssertionError: -100.0 != -1000.0 within 7 places (900.0 difference)

----------------------------------------------------------------------
Ran 1 test in 0.001s

FAILED (failures=1)


<unittest.runner.TextTestResult run=1 errors=0 failures=1>

## 8. Organizing Tests in a Project

Common layout:

```text
project/
  app/
    __init__.py
    pricing.py
    orders.py
    positions.py
  tests/
    __init__.py
    test_pricing.py
    test_orders.py
    test_positions.py
```

Example `tests/test_pricing.py`:

```python
import unittest
from app.pricing import mid_price, pnl


class TestPricing(unittest.TestCase):
    def test_mid_price(self):
        self.assertEqual(mid_price(99, 101), 100)

    def test_pnl(self):
        self.assertAlmostEqual(pnl(100, 110, 10), 100)


if __name__ == "__main__":
    unittest.main()
```

Test discovery:

```bash
python -m unittest            # discovers tests in test_*.py under cwd
python -m unittest tests.test_pricing
python -m unittest discover -s tests -p "test_*.py"
```

## 9. Running Tests: CLI, Verbosity, and Integration with pytest

Running tests with `unittest` CLI:

```bash
python -m unittest                # run all discovered tests
python -m unittest -v             # verbose
python -m unittest tests.test_all # specific module
```

You can also build **suites** programmatically (as we did in the notebook) and run
them with different **verbosity** levels.

Many projects now use **pytest**, but you can still keep `unittest`-style tests:

- pytest will discover and run `unittest.TestCase` subclasses.
- You can gradually add pytest-style tests and use its powerful fixtures and
  parametrization while reusing existing `unittest` modules.

## 10. Best Practices for Quant/Trading Code

- **Test core math heavily**:
  - Pricing formulas, indicators, risk metrics, PnL.
  - Boundary cases: zero qty, negative prices (guarded), extreme moves.
- **Isolate external dependencies**:
  - Wrap HTTP APIs, DB calls, and message buses behind small functions/classes.
  - Mock those wrappers in tests to keep them fast and deterministic.
- **Use fixtures wisely**:
  - `setUp`/`tearDown` for per-test setup.
  - `setUpClass` for expensive resources (only when safe to share).
- **Write small, focused tests**:
  - One behavior per test method; clear name (`test_...`) describing intent.
- **Keep tests close to code**:
  - Mirror your app modules in `tests/` so it’s obvious where to add new tests.
- **Automate test runs**:
  - Integrate `python -m unittest` or pytest into your CI pipeline.
- **Combine with other tools**:
  - For property-based testing of numeric code, look at `hypothesis`.
  - For coverage, use `coverage.py`.

A strong `unittest` suite around your pricing, PnL, and risk logic gives you a
safety net when refactoring or optimizing (e.g., moving hot paths to Numba, Cython,
pybind11), ensuring behavior stays correct while performance improves.